# 03. Protocol collapse on D1 (E4)

Fixed data, features and models; evaluation protocol varied.

- Protocols: P1 naive random / P2 sensor-independent / P3 site-independent / P4 site + temporal
- Tasks (same D1 rows): five-class, leak vs. non-leak, outdoor vs. indoor
- Factorial: 4 models × 4 protocols × 10 seeds = 160 fits per task (480 total)
- Additional analyses: E4b paired OOF comparison, P5 (known sensors, future period), matched training size, variance attribution
- Output: `results/E4_collapse.csv` (resumable)

In [ ]:
import sys, os, json, time
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation as ev, sfb, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
import warnings; warnings.filterwarnings("ignore", category=UserWarning)

X1, X1_max, M1, HZ = io_utils.load_d1()
y5   = M1.cls.to_numpy()
site = M1.site.to_numpy(); sid = M1.sid.to_numpy()
dt   = pd.to_datetime(M1.dt).to_numpy()

OUT = cfg.RESULTS / "E4_collapse.csv"
SUBSAMPLE = cfg.D1_SUBSAMPLE   # max training rows (30,000)
SEEDS = range(cfg.N_SEEDS)
HEADLINE = "gbdt"

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000


## 1. Tasks and protocols

In [2]:
TASKS = {
    "T5_five_class":      (np.ones(len(y5), bool), y5),
    "TL_leak_vs_nonleak": (np.ones(len(y5), bool),
                           np.where(np.isin(y5, ["out", "in"]), "leak", "nonleak")),
    "TR_out_vs_in":       (np.isin(y5, ["out", "in"]), y5),
}
for t in TASKS:
    print(f"{t:20s} {cfg.D1_TASKS[t]}")
print(f"\nfactorial: {len(ev.MODELS)} models x {len(ev.PROTOCOLS)} protocols x "
      f"{cfg.N_SEEDS} seeds = {len(ev.MODELS)*len(ev.PROTOCOLS)*cfg.N_SEEDS} fits per task, "
      f"{len(ev.MODELS)*len(ev.PROTOCOLS)*cfg.N_SEEDS*len(TASKS)} in total")

REQUIRED = {"bacc", "mf1", "cm", "recall", "n_site_shared", "n_train_available"}
done = set()
if OUT.exists():
    _d = pd.read_csv(OUT)
    if not REQUIRED <= set(_d.columns):
        OUT.rename(cfg.RESULTS / "E4_collapse_v1.csv")
        print("v1 results archived as E4_collapse_v1.csv; re-running with the new metrics")
    else:
        done = set(zip(_d.task, _d.protocol, _d.model, _d.seed))
        print(f"resuming: {len(done)} results already on disk")

T5_five_class        five classes: out / in / noise / other / normal
TL_leak_vs_nonleak   leak (out, in) vs non-leak (noise, other, normal)
TR_out_vs_in         outdoor vs indoor, leak rows only

factorial: 4 models x 4 protocols x 10 seeds = 160 fits per task, 480 in total


## 2. Model fitting (resumable)

Per fit: metrics, split inventory; class-level values stored as JSON strings

In [3]:
def run_one(tname, proto, kind, seed, mask, yfull, subsample=SUBSAMPLE, n_match=None):
    Xs, ys = X1[mask], yfull[mask]
    gsite, gsid, gdt = site[mask], sid[mask], dt[mask]
    sp = ev.get_split(proto, ys, gsite, gsid, gdt, seed)
    if sp is None:
        return None, "test fold < 30 rows"
    a, b = sp
    if len(b) < 30 or len(np.unique(ys[a])) < 2 or len(np.unique(ys[b])) < 2:
        return None, "single-class fold or < 30 test rows"
    inv = ev.split_inventory(a, b, ys, gsite, gsid, gdt)
    cap = n_match if n_match is not None else subsample
    a_used = np.random.RandomState(seed).choice(a, cap, replace=False) if len(a) > cap else a
    r = ev.evaluate(Xs[a_used], ys[a_used], Xs[b], ys[b], kind, seed, return_pred=True)
    pred = r.pop("pred")
    rec = dict(task=tname, protocol=proto, model=kind, seed=seed,
               n_train=len(a_used), n_train_available=len(a), n_test=len(b),
               n_excluded=inv["n_excluded"],
               n_site_train=inv["n_site_train"], n_site_test=inv["n_site_test"],
               n_site_shared=inv["n_site_shared"],
               n_sensor_train=inv["n_sensor_train"], n_sensor_test=inv["n_sensor_test"],
               n_sensor_shared=inv["n_sensor_shared"],
               train_dates=inv["train_dates"], test_dates=inv["test_dates"],
               p_train=json.dumps({k[8:]: v for k, v in inv.items() if k.startswith("p_train_")}),
               p_test=json.dumps({k[7:]: v for k, v in inv.items() if k.startswith("p_test_")}),
               acc=r["acc"], bacc=r["bacc"], mf1=r["mf1"], base=r["base"],
               base_trainrule=r["base_trainrule"], excess=r["excess"], cm=r["cm"],
               recall=json.dumps({k[4:]: v for k, v in r.items() if k.startswith("rec_")}),
               support=json.dumps({k[2:]: v for k, v in r.items()
                                   if k.startswith("n_") and k[2:] in set(ys)}))
    return (rec, (b, ys[b] == pred, gsite[b])), "ok"


PRED_CACHE = cfg.CACHE / "E4_gbdt_test_correctness.npz"
pred_store = dict(np.load(PRED_CACHE, allow_pickle=True)) if PRED_CACHE.exists() else {}
skipped = []
t0 = time.time()
for tname, (mask, yfull) in TASKS.items():
    for kind in ev.MODELS:
        for seed in SEEDS:
            for proto in ev.PROTOCOLS:
                if (tname, proto, kind, seed) in done:
                    continue
                res, why = run_one(tname, proto, kind, seed, mask, yfull)
                if res is None:
                    skipped.append(dict(task=tname, protocol=proto, model=kind, seed=seed, reason=why))
                    continue
                rec, (b, ok, gs) = res
                pd.DataFrame([rec]).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
                if kind == HEADLINE:
                    key = f"{tname}|{proto}|{seed}"
                    pred_store[key + "|idx"], pred_store[key + "|ok"] = b, ok
                    pred_store[key + "|site"] = gs
                print(f"  {tname:20s} {proto:18s} {kind:6s} s{seed} "
                      f"acc={rec['acc']:.4f} bacc={rec['bacc']:.4f} base={rec['base']:.4f}",
                      flush=True)
    np.savez_compressed(PRED_CACHE, **pred_store)
pd.DataFrame(skipped, columns=["task", "protocol", "model", "seed", "reason"]) \
    .to_csv(cfg.RESULTS / "E4_skipped_fits.csv", index=False)
print(f"\nelapsed {time.time() - t0:.0f} s; skipped fits: {len(skipped)}")

  T5_five_class        P1_naive_random    gbdt   s0 acc=0.8930 bacc=0.8589 base=0.3149
  T5_five_class        P2_sensor_indep    gbdt   s0 acc=0.5425 bacc=0.4239 base=0.3126
  T5_five_class        P3_site_indep      gbdt   s0 acc=0.4787 bacc=0.4224 base=0.3879
  T5_five_class        P4_site_temporal   gbdt   s0 acc=0.4202 bacc=0.3569 base=0.2856
  T5_five_class        P1_naive_random    gbdt   s1 acc=0.8927 bacc=0.8546 base=0.3149
  T5_five_class        P2_sensor_indep    gbdt   s1 acc=0.5545 bacc=0.4261 base=0.3407
  T5_five_class        P3_site_indep      gbdt   s1 acc=0.3160 bacc=0.2540 base=0.4620
  T5_five_class        P4_site_temporal   gbdt   s1 acc=0.4433 bacc=0.3736 base=0.2468
  T5_five_class        P1_naive_random    gbdt   s2 acc=0.8947 bacc=0.8590 base=0.3149
  T5_five_class        P2_sensor_indep    gbdt   s2 acc=0.5214 bacc=0.4149 base=0.2968
  T5_five_class        P3_site_indep      gbdt   s2 acc=0.5851 bacc=0.3869 base=0.4614
  T5_five_class        P4_site_temporal   g

## 3. Summary by task × model × protocol

`acc_sd`: SD over 10 seeds (split variability, not a site-level interval)

In [4]:
d4 = pd.read_csv(OUT)
s4 = (d4.groupby(["task", "model", "protocol"])
        .agg(acc=("acc", "mean"), acc_sd=("acc", "std"),
             acc_min=("acc", "min"), acc_max=("acc", "max"),
             bacc=("bacc", "mean"), bacc_sd=("bacc", "std"),
             mf1=("mf1", "mean"), base=("base", "mean"),
             excess=("excess", "mean"), excess_sd=("excess", "std"),
             n_train=("n_train", "mean"), n_test=("n_test", "mean"),
             n_seeds=("acc", "size"))
        .reset_index())
order = {p: i for i, p in enumerate(ev.PROTOCOLS)}
s4 = s4.sort_values(["task", "model", "protocol"],
                    key=lambda c: c.map(order) if c.name == "protocol" else c)
s4.to_csv(cfg.RESULTS / "E4_collapse_summary.csv", index=False)
print(s4.drop(columns=["acc_min", "acc_max", "n_train", "n_test"]).round(4).to_string(index=False))

              task  model         protocol    acc  acc_sd   bacc  bacc_sd    mf1   base  excess  excess_sd  n_seeds
     T5_five_class   gbdt  P1_naive_random 0.8935  0.0018 0.8572   0.0029 0.8702 0.3149  0.5786     0.0018       10
     T5_five_class   gbdt  P2_sensor_indep 0.5435  0.0168 0.4237   0.0083 0.4173 0.3156  0.2279     0.0091       10
     T5_five_class   gbdt    P3_site_indep 0.4806  0.0949 0.3732   0.0503 0.3542 0.4289  0.0517     0.1050       10
     T5_five_class   gbdt P4_site_temporal 0.3272  0.1104 0.3376   0.0660 0.2657 0.3406 -0.0134     0.1467       10
     T5_five_class logreg  P1_naive_random 0.6277  0.0028 0.5277   0.0029 0.5317 0.3149  0.3127     0.0028       10
     T5_five_class logreg  P2_sensor_indep 0.5132  0.0155 0.4034   0.0088 0.3892 0.3156  0.1976     0.0070       10
     T5_five_class logreg    P3_site_indep 0.4655  0.0979 0.3642   0.0409 0.3409 0.4289  0.0366     0.0964       10
     T5_five_class logreg P4_site_temporal 0.3026  0.0920 0.3229   0.067

## 4. Per-class recall and operational rates

Confusion matrices summed over seeds; leak vs. non-leak: missed leaks and unnecessary investigations per 1,000 records

In [5]:
def sum_cm(cms):
    tot, labels = None, None
    for c in cms:
        d = json.loads(c)
        m = np.asarray(d["m"])
        if tot is None:
            tot, labels = m.copy(), d["labels"]
        else:
            tot = tot + m
    return labels, tot

rec_rows, ops = [], []
for (t, m, p), g in d4.groupby(["task", "model", "protocol"]):
    labels, cm = sum_cm(g.cm)
    rec = cm.diagonal() / np.maximum(cm.sum(1), 1)
    rec_rows.append(dict(task=t, model=m, protocol=p,
                         **{f"recall_{l}": r for l, r in zip(labels, rec)},
                         cm=json.dumps({"labels": labels, "m": cm.tolist()})))
    if t == "TL_leak_vs_nonleak":
        ops.append(dict(task=t, model=m, protocol=p,
                        **ev.operational_rates({"labels": labels, "m": cm.tolist()}, "leak")))
rec_df = pd.DataFrame(rec_rows)
rec_df.to_csv(cfg.RESULTS / "E4_per_class_recall.csv", index=False)
ops = pd.DataFrame(ops)
ops.to_csv(cfg.RESULTS / "E4_operational_rates.csv", index=False)
print(rec_df[rec_df.model == HEADLINE].drop(columns="cm").round(3).to_string(index=False))
print("\nleak vs non-leak, per 1000 records (summed over seeds):")
print(ops.round(1).to_string(index=False))

              task model         protocol  recall_in  recall_noise  recall_normal  recall_other  recall_out  recall_leak  recall_nonleak
     T5_five_class  gbdt  P1_naive_random      0.846         0.763          0.977         0.782       0.918          NaN             NaN
     T5_five_class  gbdt  P2_sensor_indep      0.333         0.196          0.955         0.102       0.528          NaN             NaN
     T5_five_class  gbdt    P3_site_indep      0.303         0.129          0.816         0.099       0.496          NaN             NaN
     T5_five_class  gbdt P4_site_temporal      0.231         0.131          0.945         0.091       0.492          NaN             NaN
TL_leak_vs_nonleak  gbdt  P1_naive_random        NaN           NaN            NaN           NaN         NaN        0.912           0.897
TL_leak_vs_nonleak  gbdt  P2_sensor_indep        NaN           NaN            NaN           NaN         NaN        0.770           0.735
TL_leak_vs_nonleak  gbdt    P3_site_indep

## 5. Protocol inventory

Train/test sizes, independent and shared groups, class proportions, excluded rows (mean over seeds)
- `n_train`: rows used (max 30,000)
- `n_train_available`: rows available before the cap

In [6]:
inv = (d4[d4.model == HEADLINE]
       .groupby(["task", "protocol"])
       .agg(n_train_available=("n_train_available", "mean"), n_train=("n_train", "mean"),
            n_test=("n_test", "mean"), n_excluded=("n_excluded", "mean"),
            n_site_train=("n_site_train", "mean"), n_site_test=("n_site_test", "mean"),
            n_site_shared=("n_site_shared", "mean"),
            n_sensor_train=("n_sensor_train", "mean"), n_sensor_test=("n_sensor_test", "mean"),
            n_sensor_shared=("n_sensor_shared", "mean"), n_seeds=("seed", "size"))
       .reset_index())
props = (d4[d4.model == HEADLINE].groupby(["task", "protocol"])
         .agg(p_train=("p_train", lambda s: json.dumps(pd.DataFrame([json.loads(x) for x in s]).mean().round(4).to_dict())),
              p_test=("p_test", lambda s: json.dumps(pd.DataFrame([json.loads(x) for x in s]).mean().round(4).to_dict())))
         .reset_index())
inv = inv.merge(props, on=["task", "protocol"])
inv = inv.sort_values(["task", "protocol"], key=lambda c: c.map(order) if c.name == "protocol" else c)
inv.to_csv(cfg.RESULTS / "E4_protocol_inventory.csv", index=False)
pd.set_option("display.max_colwidth", 120)
print(inv.round(1).to_string(index=False))

              task         protocol  n_train_available  n_train  n_test  n_excluded  n_site_train  n_site_test  n_site_shared  n_sensor_train  n_sensor_test  n_sensor_shared  n_seeds                                                                           p_train                                                                            p_test
     T5_five_class  P1_naive_random            49268.0  30000.0 21116.0         0.0          48.4         45.9           45.3          2593.0         2314.4           2197.4       10 {"normal": 0.3149, "out": 0.2803, "in": 0.2121, "other": 0.1122, "noise": 0.0804} {"normal": 0.3149, "out": 0.2804, "in": 0.2122, "other": 0.1122, "noise": 0.0804}
     T5_five_class  P2_sensor_indep            49219.2  30000.0 21164.8         0.0          45.7         38.5           35.2          1897.0          813.0              0.0       10 {"normal": 0.3159, "out": 0.2831, "in": 0.2096, "other": 0.1088, "noise": 0.0826}  {"normal": 0.3132, "out": 0.2737, "in": 

## 6. Step decomposition (accuracy, BA, excess over majority)

Interval unit: seed (split variability). Site-level intervals in section 9 (E4b).

In [7]:
rng = np.random.RandomState(0)
steps = []
for (t, m), g in d4.groupby(["task", "model"]):
    for p_from, p_to in zip(ev.PROTOCOLS[:-1], ev.PROTOCOLS[1:]):
        for metric in ["acc", "excess", "bacc"]:
            a = g[g.protocol == p_from][metric].to_numpy()
            b = g[g.protocol == p_to][metric].to_numpy()
            if not len(a) or not len(b):
                continue
            boot = np.array([rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean()
                             for _ in range(cfg.N_BOOT)])
            steps.append(dict(task=t, model=m, step=f"{p_from[:2]}->{p_to[:2]}", metric=metric,
                              decline=a.mean() - b.mean(), lo=np.percentile(boot, 2.5),
                              hi=np.percentile(boot, 97.5), unit="seed (split)"))
steps = pd.DataFrame(steps)
steps.to_csv(cfg.RESULTS / "E4_step_decomposition.csv", index=False)
print(steps[steps.model == HEADLINE].pivot_table(index=["task", "step"], columns="metric",
                                                 values="decline").round(4).to_string())
for t in TASKS:
    for _, r in steps[(steps.task == t) & (steps.model == HEADLINE)].iterrows():
        rp.record(f"D1.E4.{t}.{HEADLINE}.drop_{r.step}.{r.metric}", r.decline, "E4_step_decomposition.csv",
                  "R1-M(D1 ladder),R1-minor(P1->P2)", lo=r.lo, hi=r.hi, note="unit=seed")
for _, r in s4[s4.model == HEADLINE].iterrows():
    rp.record(f"D1.E4.{r.task}.{HEADLINE}.{r.protocol}.acc", r.acc, "E4_collapse_summary.csv",
              "R1-M7", note=f"sd over seeds={r.acc_sd:.4f}")
    rp.record(f"D1.E4.{r.task}.{HEADLINE}.{r.protocol}.bacc", r.bacc, "E4_collapse_summary.csv", "R1-M7,R2-16")
    rp.record(f"D1.E4.{r.task}.{HEADLINE}.{r.protocol}.base", r.base, "E4_collapse_summary.csv", "R1-minor(Fig2)")
    rp.record(f"D1.E4.{r.task}.{HEADLINE}.{r.protocol}.excess", r.excess, "E4_collapse_summary.csv", "R1-M(excess)")

metric                        acc    bacc  excess
task               step                          
T5_five_class      P1->P2  0.3500  0.4335  0.3507
                   P2->P3  0.0629  0.0505  0.1762
                   P3->P4  0.1534  0.0356  0.0651
TL_leak_vs_nonleak P1->P2  0.1524  0.1522  0.1621
                   P2->P3  0.0239  0.0377  0.1491
                   P3->P4  0.1318  0.1283  0.0775
TR_out_vs_in       P1->P2  0.3667  0.3742  0.3706
                   P2->P3  0.0177  0.0098  0.0377
                   P3->P4  0.0197  0.0043 -0.0073


## 7. Between-model spread by protocol

In [8]:
spread = (s4.groupby(["task", "protocol"]).acc.agg(lambda v: v.max() - v.min())
          .rename("between_model_spread").reset_index())
spread.to_csv(cfg.RESULTS / "E4_between_model_spread.csv", index=False)
print(spread.pivot(index="task", columns="protocol", values="between_model_spread")
      [ev.PROTOCOLS].round(4).to_string())
for _, r in spread.iterrows():
    rp.record(f"D1.E4.{r.task}.{r.protocol}.between_model_spread", r.between_model_spread,
              "E4_between_model_spread.csv", "consistency(l.313)")

protocol            P1_naive_random  P2_sensor_indep  P3_site_indep  P4_site_temporal
task                                                                                 
T5_five_class                0.2659           0.0303         0.0151            0.0246
TL_leak_vs_nonleak           0.1501           0.0318         0.0286            0.0197
TR_out_vs_in                 0.2691           0.0340         0.0163            0.0126


## 8. Site-level bootstrap (GBDT, seed 0)

Test rows resampled by site

In [9]:
pred_store = dict(np.load(PRED_CACHE, allow_pickle=True))
cl_rows = []
for t, (mask, yfull) in TASKS.items():
    for p in ev.PROTOCOLS:
        key = f"{t}|{p}|0"
        if key + "|ok" not in pred_store:
            continue
        ci = sfb.cluster_bootstrap_acc(pred_store[key + "|ok"], pred_store[key + "|site"],
                                       n_boot=cfg.N_BOOT)
        cl_rows.append(dict(task=t, protocol=p, seed=0, acc=ci["acc"], lo=ci["lo"], hi=ci["hi"],
                            n_test_rows=ci["n_units"], n_test_sites=ci["n_clusters"],
                            unit="site"))
cl = pd.DataFrame(cl_rows)
cl.to_csv(cfg.RESULTS / "E4_site_cluster_ci_seed0.csv", index=False)
print(cl.round(4).to_string(index=False))

              task         protocol  seed    acc     lo     hi  n_test_rows  n_test_sites unit
     T5_five_class  P1_naive_random     0 0.8930 0.8516 0.9237        21116            46 site
     T5_five_class  P2_sensor_indep     0 0.5425 0.3745 0.7008        21596            40 site
     T5_five_class    P3_site_indep     0 0.4787 0.3076 0.7096        19246            15 site
     T5_five_class P4_site_temporal     0 0.4202 0.2035 0.7902         4117             9 site
TL_leak_vs_nonleak  P1_naive_random     0 0.9044 0.8694 0.9329        21116            46 site
TL_leak_vs_nonleak  P2_sensor_indep     0 0.7825 0.6952 0.8618        21596            40 site
TL_leak_vs_nonleak    P3_site_indep     0 0.7273 0.5902 0.8536        19246            15 site
TL_leak_vs_nonleak P4_site_temporal     0 0.6177 0.4468 0.8864         4117             9 site
      TR_out_vs_in  P1_naive_random     0 0.9110 0.8960 0.9244        10399            43 site
      TR_out_vs_in  P2_sensor_indep     0 0.5530 0

## 9. E4b: paired out-of-fold comparison (P1, P2, P3)

- 5-fold OOF: P1 stratified by row / P2 grouped by sensor / P3 grouped by site
- Same rows under every protocol → paired site-level bootstrap of the difference
- P4 not included (not a partition of the rows)
- Training folds capped at 30,000 rows

In [10]:
E4B = cfg.RESULTS / "E4b_paired_oof.csv"
E4B_PRED = cfg.CACHE / "E4b_oof_predictions.npz"
oof_store = dict(np.load(E4B_PRED, allow_pickle=True)) if E4B_PRED.exists() else {}
schemes = {"P1_naive_random": ("stratified", None),
           "P2_sensor_indep": ("stratified_group", sid),
           "P3_site_indep": ("stratified_group", site)}
for t, (mask, yfull) in TASKS.items():
    ys = yfull[mask].astype(str)
    for p, (scheme, g) in schemes.items():
        if f"{t}|{p}|pred" in oof_store:
            continue
        folds = ev.make_folds(scheme, ys, None if g is None else g[mask], 5, seed=0)
        pred = np.empty(len(ys), dtype=object)
        for k, (tr, te) in enumerate(folds):
            a = np.random.RandomState(k).choice(tr, SUBSAMPLE, replace=False) if len(tr) > SUBSAMPLE else tr
            pred[te] = ev.make_model(HEADLINE, 0).fit(X1[mask][a], ys[a]).predict(X1[mask][te])
        oof_store[f"{t}|{p}|pred"] = pred.astype(str)
        oof_store[f"{t}|{p}|rule"] = ev.fold_majority_rule(ys, folds)
        np.savez_compressed(E4B_PRED, **oof_store)
        print(f"  {t:20s} {p:16s} acc={np.mean(pred.astype(str) == ys):.4f}", flush=True)

rows = []
for t, (mask, yfull) in TASKS.items():
    ys = yfull[mask].astype(str)
    for p in schemes:
        pr = oof_store[f"{t}|{p}|pred"]
        s = ev.score(ys, pr)
        rows.append(dict(task=t, comparison=p, acc=s["acc"], bacc=s["bacc"],
                         rule=float(oof_store[f"{t}|{p}|rule"].mean())))
    for pa, pb in [("P1_naive_random", "P2_sensor_indep"), ("P2_sensor_indep", "P3_site_indep"),
                   ("P1_naive_random", "P3_site_indep")]:
        r = sfb.paired_cluster_bootstrap_metrics(
            ys, oof_store[f"{t}|{pa}|pred"], oof_store[f"{t}|{pb}|pred"], site[mask],
            n_boot=cfg.N_BOOT, base_a=oof_store[f"{t}|{pa}|rule"], base_b=oof_store[f"{t}|{pb}|rule"])
        rows.append(dict(task=t, comparison=f"{pa[:2]}->{pb[:2]}", **r))
        for mtr in ["acc", "bacc", "excess"]:
            rp.record(f"D1.E4b.{t}.drop_{pa[:2]}->{pb[:2]}.{mtr}", r[f"d_{mtr}"], "E4b_paired_oof.csv",
                      "R1-M8,R2-11", lo=r[f"d_{mtr}_lo"], hi=r[f"d_{mtr}_hi"], note="unit=site")
e4b = pd.DataFrame(rows)
e4b.to_csv(E4B, index=False)
print(e4b.round(4).to_string(index=False))

  T5_five_class        P1_naive_random  acc=0.8921
  T5_five_class        P2_sensor_indep  acc=0.5413
  T5_five_class        P3_site_indep    acc=0.5202
  TL_leak_vs_nonleak   P1_naive_random  acc=0.9038
  TL_leak_vs_nonleak   P2_sensor_indep  acc=0.7604
  TL_leak_vs_nonleak   P3_site_indep    acc=0.7446
  TR_out_vs_in         P1_naive_random  acc=0.9123
  TR_out_vs_in         P2_sensor_indep  acc=0.5561
  TR_out_vs_in         P3_site_indep    acc=0.5216
              task      comparison    acc   bacc   rule  n_units  n_clusters  d_acc  d_acc_lo  d_acc_hi d_acc_excludes_zero  d_bacc  d_bacc_lo  d_bacc_hi d_bacc_excludes_zero  d_excess  d_excess_lo  d_excess_hi d_excess_excludes_zero
     T5_five_class P1_naive_random 0.8921 0.8549 0.3149      NaN         NaN    NaN       NaN       NaN                 NaN     NaN        NaN        NaN                  NaN       NaN          NaN          NaN                    NaN
     T5_five_class P2_sensor_indep 0.5413 0.4164 0.3149      NaN         

## 10. P5: known sensors, future period

Train: first 70% of the period / Test: last 30%, same sensors
Single-class sensors → identity lookup also solves this setting

In [11]:
p5 = []
for t, (mask, yfull) in TASKS.items():
    ys = yfull[mask]
    sp = ev.get_split("P5_temporal_known", ys, site[mask], sid[mask], dt[mask], 0)
    if sp is None:
        continue
    a, b = sp
    inv5 = ev.split_inventory(a, b, ys, site[mask], sid[mask], dt[mask])
    look = sfb._lookup_predict(sid[mask][a], ys[a], sid[mask][b],
                               pd.Series(ys[a]).value_counts().idxmax())
    s_look = ev.score(ys[b], look)
    for seed in SEEDS:
        a_used = np.random.RandomState(seed).choice(a, SUBSAMPLE, replace=False) if len(a) > SUBSAMPLE else a
        r = ev.evaluate(X1[mask][a_used], ys[a_used], X1[mask][b], ys[b], HEADLINE, seed)
        p5.append(dict(task=t, seed=seed, acc=r["acc"], bacc=r["bacc"], base=r["base"],
                       sensor_lookup_acc=s_look["acc"], sensor_lookup_bacc=s_look["bacc"],
                       n_train_available=len(a), n_test=len(b),
                       n_sensor_test=inv5["n_sensor_test"], n_sensor_shared=inv5["n_sensor_shared"],
                       train_dates=inv5["train_dates"], test_dates=inv5["test_dates"]))
p5 = pd.DataFrame(p5)
p5.to_csv(cfg.RESULTS / "E4_P5_temporal_known.csv", index=False)
p5s = p5.groupby("task").agg(acc=("acc", "mean"), acc_sd=("acc", "std"), bacc=("bacc", "mean"),
                             base=("base", "first"), sensor_lookup_acc=("sensor_lookup_acc", "first"),
                             n_test=("n_test", "first"), n_sensor_test=("n_sensor_test", "first"))
print(p5s.round(4).to_string())
for t, r in p5s.iterrows():
    rp.record(f"D1.P5.{t}.acc", r.acc, "E4_P5_temporal_known.csv", "R2-2")
    rp.record(f"D1.P5.{t}.sensor_lookup_acc", r.sensor_lookup_acc, "E4_P5_temporal_known.csv", "R2-2")

                       acc  acc_sd    bacc    base  sensor_lookup_acc  n_test  n_sensor_test
task                                                                                        
T5_five_class       0.7930  0.0102  0.6581  0.4917                1.0    3185            225
TL_leak_vs_nonleak  0.8476  0.0088  0.8091  0.7074                1.0    3185            225
TR_out_vs_in        0.7383  0.0000  0.7475  0.5327                1.0    3137             94


## 11. Matched training size

Training rows per protocol fixed to the smallest size available across protocols (per task)

In [12]:
MATCH = cfg.RESULTS / "E4_matched_training_size.csv"
mt = pd.read_csv(MATCH) if MATCH.exists() else pd.DataFrame()
done_m = set(zip(mt.task, mt.protocol, mt.seed)) if len(mt) else set()
for t, (mask, yfull) in TASKS.items():
    n_match = int(d4[(d4.task == t)].n_train_available.min())
    n_match = min(n_match, SUBSAMPLE)
    for seed in range(cfg.N_SEEDS_AUX):
        for p in ev.PROTOCOLS:
            if (t, p, seed) in done_m:
                continue
            res, why = run_one(t, p, HEADLINE, seed, mask, yfull, n_match=n_match)
            if res is None:
                continue
            rec = res[0]
            pd.DataFrame([dict(task=t, protocol=p, seed=seed, n_match=n_match, n_train=rec["n_train"],
                               acc=rec["acc"], bacc=rec["bacc"], base=rec["base"], excess=rec["excess"])]) \
                .to_csv(MATCH, mode="a", header=not MATCH.exists(), index=False)
mt = pd.read_csv(MATCH)
mts = mt.groupby(["task", "protocol"]).agg(n_match=("n_match", "first"), acc=("acc", "mean"),
                                           acc_sd=("acc", "std"), bacc=("bacc", "mean"),
                                           excess=("excess", "mean")).reset_index()
ref = s4[s4.model == HEADLINE][["task", "protocol", "acc", "n_train"]].rename(
    columns={"acc": "acc_unmatched", "n_train": "n_train_unmatched"})
mts = mts.merge(ref, on=["task", "protocol"])
mts = mts.sort_values(["task", "protocol"], key=lambda c: c.map(order) if c.name == "protocol" else c)
mts.to_csv(cfg.RESULTS / "E4_matched_training_size_summary.csv", index=False)
print(mts.round(4).to_string(index=False))

              task         protocol  n_match    acc  acc_sd   bacc  excess  acc_unmatched  n_train_unmatched
     T5_five_class  P1_naive_random    28728 0.8921  0.0019 0.8550  0.5772         0.8935            30000.0
     T5_five_class  P2_sensor_indep    28728 0.5436  0.0181 0.4233  0.2233         0.5435            30000.0
     T5_five_class    P3_site_indep    28728 0.4492  0.0964 0.3688  0.0705         0.4806            30000.0
     T5_five_class P4_site_temporal    28728 0.3247  0.0936 0.3138  0.0112         0.3272            29777.3
TL_leak_vs_nonleak  P1_naive_random    28728 0.9036  0.0024 0.9038  0.3961         0.9046            30000.0
TL_leak_vs_nonleak  P2_sensor_indep    28728 0.7605  0.0208 0.7609  0.2364         0.7521            30000.0
TL_leak_vs_nonleak    P3_site_indep    28728 0.7075  0.0505 0.6891  0.0885         0.7283            30000.0
TL_leak_vs_nonleak P4_site_temporal    28728 0.6027  0.0826 0.5962  0.0080         0.5965            29777.3
      TR_out_vs_in 

## 12. Variance attribution (protocol vs. model)

- Two-way decomposition with replication, per task (`evaluation.factorial_ss`)
- D1 only; the three tasks share the same rows (not independent replications)
- Shares are descriptive for these models and protocols only

In [13]:
va_rows, cells = [], []
for task, g in d4.groupby("task"):
    tab, cm_ = ev.factorial_ss(g, "protocol", "model", "acc")
    tab.insert(0, "task", task); cm_.insert(0, "task", task)
    va_rows.append(tab); cells.append(cm_)
va = pd.concat(va_rows, ignore_index=True)
cells = pd.concat(cells, ignore_index=True)
va.to_csv(cfg.RESULTS / "E4_variance_attribution.csv", index=False)
cells.to_csv(cfg.RESULTS / "E4_cell_means.csv", index=False)
print(va.drop(columns=["ss_total"]).round(4).to_string(index=False))
for task, g in va.groupby("task"):
    for _, r in g.iterrows():
        rp.record(f"D1.factorial.{task}.{r.term}.share", r.ss_share, "E4_variance_attribution.csv",
                  "R1-M5,R1-M6,R2-9", note=f"n_fits={r.n_fits}, balanced={r.balanced}")
print("\nfits per task:", va.groupby("task").n_fits.first().to_dict())

              task             term     ss  df  ss_share  partial_eta2  identity_residual  balanced  n_fits  reps_per_cell
     T5_five_class         protocol 5.2938   3    0.8095        0.8761               -0.0      True     160             10
     T5_five_class            model 0.1661   3    0.0254        0.1816               -0.0      True     160             10
     T5_five_class protocol x model 0.3313   9    0.0507        0.3069               -0.0      True     160             10
     T5_five_class  residual (seed) 0.7484 144    0.1144           NaN               -0.0      True     160             10
TL_leak_vs_nonleak         protocol 1.7711   3    0.7913        0.8478               -0.0      True     160             10
TL_leak_vs_nonleak            model 0.0563   3    0.0252        0.1505               -0.0      True     160             10
TL_leak_vs_nonleak protocol x model 0.0927   9    0.0414        0.2258               -0.0      True     160             10
TL_leak_vs_nonle